# Load-Aware SAGE: Adaptive Retrieval Baseline

This notebook provides the **reproducible baseline retrieval pipeline** for the SAGE (Self-Adaptive Generation & Estimation) research project:
- **Corpus**: Wikipedia articles chunked via `RecursiveCharacterTextSplitter` (chunk size = 500, overlap = 100)
- **Dense Retrieval**: BGE-M3 (`BAAI/bge-m3`) with FAISS `IndexFlatIP` (normalized inner product)
- **Sparse Retrieval**: BM25Okapi
- **Hybrid Fusion**: Reciprocal Rank Fusion (RRF, constant $k=60$)
- **Evaluation**: Multi-query Hit@K and Mean Reciprocal Rank (MRR) across candidate budgets $K \in [1, 2, 3, 5, 7, 10, 15, 20, 25, 30]$


In [ ]:
import sys
from pathlib import Path

# Automatically add repository root to Python path
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.utils.logger import setup_logger, set_seed

# Enforce deterministic random seed across NumPy, PyTorch, and Python
set_seed(config.RANDOM_STATE)
logger = setup_logger("sage_baseline_notebook")

print("Project Root:", config.PROJECT_ROOT)
print("Candidate K values:", config.CANDIDATE_K_VALUES)
print("Target P95 Latency SLO:", config.SLO_P95_LATENCY_SECONDS, "s")


## 1. Load Preprocessed Corpus & Question Records
We load the preprocessed and deduplicated Natural Questions sample documents and ground-truth question records from `artifacts/dataset/`.


In [ ]:
from src.data.loader import NQDataLoader

loader = NQDataLoader()
documents = loader.load_documents()
questions = loader.load_questions()

# Separate answerable questions for unbiased retrieval evaluation
eval_questions = [q for q in questions if q.has_answer]

print(f"Total Unique Documents: {len(documents)}")
print(f"Total Question Records: {len(questions)}")
print(f"Evaluation Questions with Ground Truth: {len(eval_questions)}")
print(f"Sample Question: '{eval_questions[0].question}' -> Ground Truth: {eval_questions[0].short_answers}")


## 2. Document Chunking
We apply recursive character chunking with `chunk_size=500` and `chunk_overlap=100`, matching the experimental protocol. Chunks are cached to `artifacts/dataset/chunks.jsonl`.


In [ ]:
import json
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=config.CHUNK_SIZE,
    chunk_overlap=config.CHUNK_OVERLAP,
    separators=config.CHUNK_SEPARATORS,
    length_function=len,
    is_separator_regex=False
)

# Generate chunks across all unique documents
all_chunks = []
for doc in documents:
    doc_splits = text_splitter.split_text(doc.text)
    for i, split in enumerate(doc_splits):
        all_chunks.append({
            "chunk_id": f"{doc.doc_id}_chunk_{i+1}",
            "doc_id": doc.doc_id,
            "document_title": doc.title,
            "chunk_text": split
        })

# Cache chunks to disk for reproducibility
with open(config.CHUNKS_PATH, "w", encoding="utf-8") as f:
    for chunk in all_chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + "\n")

chunk_lengths = [len(c["chunk_text"]) for c in all_chunks]
print(f"Total Chunks: {len(all_chunks)}")
print(f"Mean Chunk Length: {sum(chunk_lengths) / len(chunk_lengths):.1f} characters")
print(f"Min / Max: {min(chunk_lengths)} / {max(chunk_lengths)} characters")
print(f"Saved chunks to: {config.CHUNKS_PATH.name}")


## 3. Dense Retrieval Indexing (BGE-M3 + FAISS)
We encode chunk texts into 1024-dimensional dense vectors using `BAAI/bge-m3`. Embeddings are cached to `artifacts/embeddings/bge_m3_embeddings.npy` and indexed with FAISS `IndexFlatIP`.


In [ ]:
import numpy as np
import faiss
from FlagEmbedding import BGEM3FlagModel

EMBEDDINGS_FILE = config.EMBEDDINGS_DIR / "bge_m3_embeddings.npy"
FAISS_INDEX_FILE = config.FAISS_INDEX_DIR / "index.faiss"

print("Loading BGE-M3 FlagModel...")
model = BGEM3FlagModel(config.DENSE_MODEL_NAME, use_fp16=config.DENSE_USE_FP16)

chunk_texts = [c["chunk_text"] for c in all_chunks]

# Check if cached embeddings exist to avoid expensive re-encoding
if EMBEDDINGS_FILE.exists():
    print(f"Loading cached embeddings from {EMBEDDINGS_FILE.name}...")
    embeddings = np.load(EMBEDDINGS_FILE)
else:
    print(f"Generating embeddings for {len(chunk_texts)} chunks...")
    embeddings = model.encode(chunk_texts, batch_size=16, max_length=512)["dense_vecs"]
    embeddings = np.asarray(embeddings, dtype=np.float32)
    np.save(EMBEDDINGS_FILE, embeddings)
    print(f"Saved embeddings to {EMBEDDINGS_FILE.name}")

print("Embeddings shape:", embeddings.shape)

# Build FAISS IndexFlatIP
dimension = embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dimension)
normalized_embeddings = embeddings.copy()
faiss.normalize_L2(normalized_embeddings)
faiss_index.add(normalized_embeddings)

# Persist FAISS index
faiss.write_index(faiss_index, str(FAISS_INDEX_FILE))
print(f"FAISS index built and saved to {FAISS_INDEX_FILE.name} with {faiss_index.ntotal} vectors.")


## 4. Sparse Retrieval Indexing (BM25Okapi)
We tokenize the corpus using regular-expression word boundaries and construct the sparse BM25Okapi index.


In [ ]:
import re
from rank_bm25 import BM25Okapi

# Regex word tokenization
tokenized_corpus = [
    re.findall(r"\b\w+\b", text.lower())
    for text in chunk_texts
]

bm25_index = BM25Okapi(tokenized_corpus)
print(f"BM25Okapi index successfully initialized over {len(tokenized_corpus)} passages.")


## 5. Retrieval Functions & Hybrid Reciprocal Rank Fusion (RRF)
We implement the retrieval functions:
1. `dense_retrieval(query, top_k)`
2. `bm25_retrieval(query, top_k)`
3. `hybrid_retrieval(query, top_k, rrf_k=60)`


In [ ]:
def dense_retrieval(query: str, top_k: int = 30) -> list[int]:
    """Retrieve top_k chunk indices using BGE-M3 and FAISS."""
    q_emb = model.encode([query], max_length=512)["dense_vecs"]
    q_emb = np.asarray(q_emb, dtype=np.float32)
    faiss.normalize_L2(q_emb)
    _, indices = faiss_index.search(q_emb, top_k)
    return indices[0].tolist()

def bm25_retrieval(query: str, top_k: int = 30) -> list[int]:
    """Retrieve top_k chunk indices using BM25Okapi."""
    tokens = re.findall(r"\b\w+\b", query.lower())
    scores = bm25_index.get_scores(tokens)
    top_indices = scores.argsort()[-top_k:][::-1]
    return top_indices.tolist()

def reciprocal_rank_fusion(dense_ids: list[int], sparse_ids: list[int], k: int = 60) -> list[tuple[int, float]]:
    """Combine dense and sparse ranked lists using Reciprocal Rank Fusion."""
    rrf_scores = {}
    for rank, chunk_id in enumerate(dense_ids, start=1):
        rrf_scores[chunk_id] = rrf_scores.get(chunk_id, 0.0) + (1.0 / (k + rank))
    for rank, chunk_id in enumerate(sparse_ids, start=1):
        rrf_scores[chunk_id] = rrf_scores.get(chunk_id, 0.0) + (1.0 / (k + rank))
    return sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

def hybrid_retrieval(query: str, top_k: int = 30, rrf_k: int = 60) -> list[int]:
    """Retrieve top_k chunk indices using RRF hybrid fusion."""
    d_ids = dense_retrieval(query, top_k=top_k)
    s_ids = bm25_retrieval(query, top_k=top_k)
    fused = reciprocal_rank_fusion(d_ids, s_ids, k=rrf_k)
    return [chunk_id for chunk_id, _ in fused[:top_k]]

# Test query execution
sample_q = eval_questions[0].question
d_res = dense_retrieval(sample_q, top_k=5)
s_res = bm25_retrieval(sample_q, top_k=5)
h_res = hybrid_retrieval(sample_q, top_k=5)

print(f"Sample Query: '{sample_q}'")
print("Dense Top 3:", d_res[:3])
print("Sparse Top 3:", s_res[:3])
print("Hybrid Top 3:", h_res[:3])


## 6. Multi-Query Retrieval Evaluation (Hit@K & MRR)
We evaluate retrieval quality across all answerable questions using **each question's actual ground-truth annotations** (replacing the hardcoded 'Persephone' answer).

Metrics:
- **Hit@K**: 1 if at least one ground-truth answer string appears in the top $K$ retrieved chunks; 0 otherwise.
- **Mean Reciprocal Rank (MRR)**: Reciprocal of the rank of the first chunk containing a ground-truth answer.


In [ ]:
import pandas as pd
from tqdm import tqdm

def evaluate_retriever_hit_and_mrr(retriever_fn, eval_data, k_list):
    """
    Computes Hit@K across all candidate K values and MRR for a given retrieval function.
    """
    max_k = max(k_list)
    hit_accum = {k: 0 for k in k_list}
    rr_total = 0.0
    num_queries = len(eval_data)

    for item in tqdm(eval_data, desc=retriever_fn.__name__):
        query = item.question
        gold_answers = [a.lower() for a in item.short_answers]
        
        ranked_chunk_ids = retriever_fn(query, top_k=max_k)
        
        # Determine the rank of the first hit
        first_hit_rank = None
        for rank, cid in enumerate(ranked_chunk_ids, start=1):
            chunk_text = all_chunks[cid]["chunk_text"].lower()
            if any(ans in chunk_text for ans in gold_answers):
                first_hit_rank = rank
                break
        
        # Accumulate Hit@K and MRR
        if first_hit_rank is not None:
            rr_total += 1.0 / first_hit_rank
            for k in k_list:
                if first_hit_rank <= k:
                    hit_accum[k] += 1

    hit_rates = {k: hit_accum[k] / num_queries for k in k_list}
    mrr = rr_total / num_queries
    return hit_rates, mrr

# Evaluate across candidate budgets
test_k_values = [1, 2, 3, 5, 7, 10, 15, 20, 25, 30]

print(f"Evaluating {len(eval_questions)} queries across K={test_k_values}...")
dense_hits, dense_mrr = evaluate_retriever_hit_and_mrr(dense_retrieval, eval_questions, test_k_values)
bm25_hits, bm25_mrr = evaluate_retriever_hit_and_mrr(bm25_retrieval, eval_questions, test_k_values)
hybrid_hits, hybrid_mrr = evaluate_retriever_hit_and_mrr(hybrid_retrieval, eval_questions, test_k_values)

# Build summary DataFrame
records = []
for k in test_k_values:
    records.append({
        "K": k,
        "Dense Hit@K": f"{dense_hits[k]*100:.1f}%",
        "BM25 Hit@K": f"{bm25_hits[k]*100:.1f}%",
        "Hybrid Hit@K": f"{hybrid_hits[k]*100:.1f}%",
    })

results_df = pd.DataFrame(records)
print("\n=== Retrieval Hit@K Performance Comparison ===")
print(results_df.to_string(index=False))
print(f"\nMRR -> Dense: {dense_mrr:.4f} | BM25: {bm25_mrr:.4f} | Hybrid (RRF): {hybrid_mrr:.4f}")


## 7. Visualization: Hit@K vs. Retrieval Budget K
We visualize the retrieval saturation curve to observe the diminishing returns of increasing $K$ (the fundamental motivation for adaptive budget allocation in SAGE).


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))
plt.plot(test_k_values, [dense_hits[k] for k in test_k_values], marker="o", label=f"Dense (BGE-M3, MRR={dense_mrr:.3f})")
plt.plot(test_k_values, [bm25_hits[k] for k in test_k_values], marker="s", label=f"Sparse (BM25, MRR={bm25_mrr:.3f})")
plt.plot(test_k_values, [hybrid_hits[k] for k in test_k_values], marker="^", linewidth=2.5, label=f"Hybrid (RRF k=60, MRR={hybrid_mrr:.3f})")

plt.title("Retrieval Hit@K vs. Retrieval Budget K (Natural Questions Sample)", fontsize=12, fontweight="bold")
plt.xlabel("Retrieval Budget (K passages)", fontsize=11)
plt.ylabel("Hit@K", fontsize=11)
plt.xticks(test_k_values)
plt.ylim(0, 1.05)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(fontsize=10)
plt.tight_layout()

# Save plot to experiments directory
chart_path = config.BASELINE_RESULTS_DIR / "retrieval_hit_at_k.png"
plt.savefig(chart_path, dpi=300)
print(f"Chart saved to {chart_path}")
plt.show()
